# Raw Reactions Pipeline Walkthrough

This notebook demonstrates the structured raw-reaction pipeline step by step: preset selection, prompt construction, LLM response parsing, self-revalidation, validation prompt construction, verdict handling, CID parsing, and final parsed reaction shape.

By default it uses deterministic demo responses and does **not** call an LLM or write pipeline outputs. Flip the safety switches in the setup cell when you intentionally want to use the live API or run the real fetch/validate/parse stages.

## 1. Setup

The notebook can be launched from the repo root or from the `notebooks/` directory. It imports the real pipeline code so every prompt/schema/parser helper shown here is the one used by the CLI.

In [1]:
from pathlib import Path
from pprint import pprint
import json
import os
import sys

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.chemy import Chemy
from scripts.ops.raw_reactions.presets import DEFAULT_PRESET_NAMES, get_preset
from scripts.ops.raw_reactions.prompts import (
    VALIDATE_BATCH_INSTRUCT,
    build_fetch_prompt,
    build_revalidate_prompt,
    format_reactions_for_validation,
)
from scripts.ops.raw_reactions.fetcher import parse_reactions_jsonl, _is_valid_reaction_obj

DATA_DIR = REPO_ROOT / "data"
chemy = Chemy(str(DATA_DIR))

# Safety switches. Keep these False for inspection-only notebook runs.
RUN_REAL_LLM = False
RUN_WRITES = False
REAL_MODEL = chemy.reaction_llm.gpt_oss

print(f"Repo root: {REPO_ROOT}")
print(f"Data dir:  {DATA_DIR}")
print(f"Live LLM calls enabled: {RUN_REAL_LLM}")
print(f"Pipeline writes enabled: {RUN_WRITES}")

Repo root: /home/canary/Documents/Code/chemy
Data dir:  /home/canary/Documents/Code/chemy/data
Live LLM calls enabled: False
Pipeline writes enabled: False


## 2. Inspect Presets

Presets decide which compounds are submitted to the LLM, what role the target compound should play, and how adventurous the prompt should be. This cell counts the current candidate set for each preset using the active data directory.

In [2]:
def preset_count(name):
    preset = get_preset(name)
    criteria = preset.build_criteria(chemy.compounds)
    matched = [chem for chem in chemy.compounds.chems if criteria(chem)]
    return {
        "preset": name,
        "total": len(matched),
        "mapped": sum(1 for chem in matched if chem["cid"] > 0),
        "unmapped": sum(1 for chem in matched if chem["cid"] < 0),
        "position": preset.position,
        "scope": preset.scope,
    }

for name in DEFAULT_PRESET_NAMES:
    pprint(preset_count(name))

{'mapped': 39291,
 'position': 'any',
 'preset': 'default_rp',
 'scope': 'documented',
 'total': 39291,
 'unmapped': 0}
{'mapped': 2698,
 'position': 'any',
 'preset': 'wiki_crc_rp',
 'scope': 'documented_less_common',
 'total': 2698,
 'unmapped': 0}
{'mapped': 1327,
 'position': 'any',
 'preset': 'simple_inorganic_wiki_rp',
 'scope': 'documented_less_common',
 'total': 1327,
 'unmapped': 0}
{'mapped': 2868,
 'position': 'any',
 'preset': 'simple_organic_wiki_rp',
 'scope': 'documented_less_common',
 'total': 2868,
 'unmapped': 0}
{'mapped': 71,
 'position': 'any',
 'preset': 'oxidizers_wiki_rp',
 'scope': 'documented_less_common',
 'total': 71,
 'unmapped': 0}
{'mapped': 233,
 'position': 'any',
 'preset': 'corrosive_wiki_rp',
 'scope': 'documented_less_common',
 'total': 233,
 'unmapped': 0}
{'mapped': 9037,
 'position': 'any',
 'preset': 'wiki_uncommon_rp',
 'scope': 'documented_less_common',
 'total': 9037,
 'unmapped': 0}
{'mapped': 277,
 'position': 'any',
 'preset': 'top_rare_rp

## 3. Select One Compound

Pick a preset and an optional compound name substring. The selected compound is what the fetch prompt will be built around. `wiki_crc_rp` is a good default because it is relatively robust: mapped compounds with both a wiki page and CRC-backed physical-property data.

In [3]:
PRESET_NAME = "wiki_crc_rp"
QUERY_NAME = "hydrogen peroxide"  # Set to None to use the first candidate.

preset = get_preset(PRESET_NAME)
criteria = preset.build_criteria(chemy.compounds)
candidates = [chem for chem in chemy.compounds.chems if criteria(chem)]

if QUERY_NAME:
    lowered = QUERY_NAME.lower()
    matches = [chem for chem in candidates if lowered in chem["cmpdname"].lower()]
    if not matches:
        raise ValueError(f"No candidate in {PRESET_NAME!r} matched {QUERY_NAME!r}")
    chem = matches[0]
else:
    chem = candidates[0]

print(f"Preset: {PRESET_NAME}")
print(f"Candidates: {len(candidates)}")
print("Selected compound:")
pprint({key: chem.get(key) for key in ["cid", "cmpdname", "mf", "wiki", "heavy_count", "bertz_complexity"]})

Preset: wiki_crc_rp
Candidates: 2698
Selected compound:
{'bertz_complexity': 0.0,
 'cid': 784,
 'cmpdname': 'Hydrogen Peroxide',
 'heavy_count': 2,
 'mf': 'H2O2',
 'wiki': 'https://en.wikipedia.org/wiki/Hydrogen_peroxide'}


## 4. Build The Fetch Prompt

The production fetcher also injects known LLM reactions for the same CID as context, up to its configured cap. This helps ask for additional reactions rather than rediscovering the same ones.

In [4]:
existing_by_cid = chemy.raw_reactions._build_existing_reactions_context()
chemy.raw_reactions._fetcher.set_existing_reactions(existing_by_cid)
existing_context = chemy.raw_reactions._fetcher._get_existing_context(chem["cid"], preset.position)[:5]

fetch_prompt = build_fetch_prompt(
    chem["cmpdname"],
    preset.position,
    preset.scope,
    existing_context,
)

print("Existing context examples:")
pprint(existing_context)
print("\n--- FETCH PROMPT SENT TO LLM ---")
print(fetch_prompt)

Existing context examples:
[]

--- FETCH PROMPT SENT TO LLM ---
Provide a list of documented chemical reactions involving Hydrogen Peroxide, where it appears as a reagent or product. Include not only the most common reactions, but also less common or unusual ones, as long as you are absolutely sure they are real and correct. Return one JSON object per line (JSONL). Each JSON object has these fields:
- "reagents": list of {"name": "<clean chemical name>", "phase": "<g|l|s|aq>"}
- "products": list of {"name": "<clean chemical name>", "phase": "<g|l|s|aq>"}
- Optional "solvent": primary solvent name or null
- Optional "catalyst": catalyst name or null
- Optional per-compound "note": short qualifier; omit it when not needed

Rules:
- Use full chemical names, not formulas.
- The name field must contain ONLY the chemical name.
- Put qualifiers such as concentrated, dilute, powdered, heated, bubbled through solution, or acidic solution in note, not in name.
- Omit note entirely when there is 

## 5. Receive And Parse Raw JSONL

The LLM must return one JSON object per line. The local parser is strict: every reagent/product needs `name` and `phase`, `phase` must be one of `g/l/s/aq`, `note` is optional, and malformed lines are skipped with stats. The demo response includes one deliberately invalid line so you can see the skip accounting.

In [5]:
demo_fetch_response = "\n".join([
    json.dumps({
        "reagents": [
            {"name": "hydrogen peroxide", "phase": "aq"},
            {"name": "potassium iodide", "phase": "aq"},
        ],
        "products": [
            {"name": "water", "phase": "l"},
            {"name": "iodine", "phase": "aq"},
            {"name": "oxygen", "phase": "g"},
        ],
        "solvent": "water",
        "catalyst": None,
    }),
    json.dumps({
        "reagents": [
            {"name": "hydrogen peroxide", "phase": "aq", "note": "acidic solution"},
            {"name": "iron(II) sulfate", "phase": "aq"},
        ],
        "products": [
            {"name": "iron(III) sulfate", "phase": "aq"},
            {"name": "water", "phase": "l"},
        ],
        "solvent": "water",
        "catalyst": None,
    }),
    # Missing phase: this should be rejected by local schema validation.
    json.dumps({
        "reagents": [{"name": "hydrogen peroxide"}],
        "products": [{"name": "water", "phase": "l"}],
    }),
])

if RUN_REAL_LLM:
    raw_response = chemy.llm_client.fetch_answer_str(fetch_prompt, REAL_MODEL)
else:
    raw_response = demo_fetch_response

print("--- RAW RESPONSE RECEIVED ---")
print(raw_response)

parse_stats = {}
raw_reactions = parse_reactions_jsonl(raw_response, parse_stats)

print("\nParse stats:")
pprint(parse_stats)
print("\nAccepted structured reactions:")
pprint(raw_reactions)

--- RAW RESPONSE RECEIVED ---
{"reagents": [{"name": "hydrogen peroxide", "phase": "aq"}, {"name": "potassium iodide", "phase": "aq"}], "products": [{"name": "water", "phase": "l"}, {"name": "iodine", "phase": "aq"}, {"name": "oxygen", "phase": "g"}], "solvent": "water", "catalyst": null}
{"reagents": [{"name": "hydrogen peroxide", "phase": "aq", "note": "acidic solution"}, {"name": "iron(II) sulfate", "phase": "aq"}], "products": [{"name": "iron(III) sulfate", "phase": "aq"}, {"name": "water", "phase": "l"}], "solvent": "water", "catalyst": null}
{"reagents": [{"name": "hydrogen peroxide"}], "products": [{"name": "water", "phase": "l"}]}

Parse stats:
{'accepted': 2, 'invalid_schema': 1}

Accepted structured reactions:
[{'catalyst': None,
  'products': [{'name': 'water', 'phase': 'l'},
               {'name': 'iodine', 'phase': 'aq'},
               {'name': 'oxygen', 'phase': 'g'}],
  'reagents': [{'name': 'hydrogen peroxide', 'phase': 'aq'},
               {'name': 'potassium iodide

## 6. Self-Revalidation Prompt

After the first successful fetch, the pipeline asks the same selected model to review and correct the structured reactions. This is where malformed phases, misplaced qualifiers, solvent/catalyst mistakes, and invalid chemistry should be corrected or removed before batch validation.

In [6]:
raw_jsonl_for_revalidation = "\n".join(json.dumps(reaction) for reaction in raw_reactions)
revalidate_prompt = build_revalidate_prompt(raw_jsonl_for_revalidation)

print("--- REVALIDATION PROMPT SENT TO LLM ---")
print(revalidate_prompt)

if RUN_REAL_LLM:
    revalidated_response = chemy.llm_client.fetch_answer_str(revalidate_prompt, REAL_MODEL)
else:
    # Demo path: pretend the model accepted the two locally valid lines unchanged.
    revalidated_response = raw_jsonl_for_revalidation

print("\n--- REVALIDATION RESPONSE RECEIVED ---")
print(revalidated_response)

revalidate_stats = {}
revalidated_reactions = parse_reactions_jsonl(revalidated_response, revalidate_stats)

print("\nRevalidation parse stats:")
pprint(revalidate_stats)
print("\nRevalidated reactions:")
pprint(revalidated_reactions)

--- REVALIDATION PROMPT SENT TO LLM ---
Review the following chemical reactions for correctness. For each, verify:
1. The reaction is chemically valid and documented.
2. Reagents and products are correctly identified.
3. Every reagent and product has a reasonable phase annotation (g, l, s, aq) for typical stated or implicit reaction conditions.
4. Solvent and catalyst (if listed) are appropriate.

Return the corrected list in the same JSONL format. Each JSON object has these fields:
- "reagents": list of {"name": "<clean chemical name>", "phase": "<g|l|s|aq>"}
- "products": list of {"name": "<clean chemical name>", "phase": "<g|l|s|aq>"}
- Optional "solvent": primary solvent name or null
- Optional "catalyst": catalyst name or null
- Optional per-compound "note": short qualifier; omit it when not needed

Remove chemically invalid reactions. Fix incorrect phases, solvents, catalysts, or misplaced name qualifiers.
Return ONLY valid JSONL lines, no extra text.

{"reagents": [{"name": "hyd

## 7. CID Mapping And Deduplication Before Validation

The validator only stages reactions that can be parsed into unique CIDs. `phase` and `note` are preserved as metadata, but reaction identity (`rid`) is based on reagent/product CIDs only.

In [7]:
staged = []
seen_rids = set()
prevalidation_report = []

for reaction_obj in revalidated_reactions:
    schema_ok = _is_valid_reaction_obj(reaction_obj)
    parsed_reaction, unmapped = chemy.reaction_llm.parse_structured_reaction(reaction_obj)
    rid = parsed_reaction["rid"] if parsed_reaction else None
    duplicate = rid in seen_rids if rid else False
    if parsed_reaction and not duplicate:
        staged.append({"cid": chem["cid"], "reaction": reaction_obj})
        seen_rids.add(rid)
    prevalidation_report.append({
        "schema_ok": schema_ok,
        "rid": rid,
        "duplicate": duplicate,
        "unmapped": sorted(unmapped),
        "parsed": parsed_reaction,
    })

print("Prevalidation report:")
pprint(prevalidation_report)
print("\nStaged reactions for validator:")
pprint(staged)

Prevalidation report:
[{'duplicate': False,
  'parsed': {'complexity': 0.4,
             'products': [{'cid': 807,
                           'norm_name': 'iodine',
                           'original_name': 'iodine',
                           'phase': 'aq'},
                          {'cid': 962,
                           'norm_name': 'water',
                           'original_name': 'water',
                           'phase': 'l'},
                          {'cid': 977,
                           'norm_name': 'oxygen',
                           'original_name': 'oxygen',
                           'phase': 'g'}],
             'reagents': [{'cid': 784,
                           'norm_name': 'hydrogenperoxide',
                           'original_name': 'hydrogen peroxide',
                           'phase': 'aq'},
                          {'cid': 4875,
                           'norm_name': 'potassiumiodide',
                           'original_name': 'potassium iodide',

## 8. Batch Validation Prompt And Verdict Parsing

The validator asks the LLM to judge both chemical validity and phase plausibility. A reaction should be marked `Invalid` if the chemistry is wrong **or** any specified phase is unreasonable.

In [8]:
validation_prompt = f"{VALIDATE_BATCH_INSTRUCT}\n{format_reactions_for_validation(staged)}"

print("--- VALIDATION PROMPT SENT TO LLM ---")
print(validation_prompt)

if RUN_REAL_LLM:
    validation_response = chemy.llm_client.fetch_answer_str(validation_prompt, REAL_MODEL)
else:
    validation_response = "\n".join(f"{i + 1}. Valid" for i in range(len(staged)))

print("\n--- VALIDATION RESPONSE RECEIVED ---")
print(validation_response)

verdicts = chemy.raw_reactions._validator._extract_verdicts(validation_response)
if len(verdicts) != len(staged):
    raise ValueError(f"Expected {len(staged)} verdicts, got {len(verdicts)}")

validated_entries = []
for staged_entry, verdict in zip(staged, verdicts):
    result = dict(staged_entry)
    result.update({
        "valid": verdict,
        "confidence": 1.0 if verdict else 0.0,
        "rounds": 1,
        "positives": int(verdict),
        "source": REAL_MODEL if RUN_REAL_LLM else "demo-validator",
    })
    validated_entries.append(result)

print("\nValidated entries:")
pprint(validated_entries)

--- VALIDATION PROMPT SENT TO LLM ---
You will be given a list of chemical reactions with phase annotations. For each reaction, determine if it is chemically valid/documented and if every specified phase is plausible for the stated or typical implicit reaction conditions.
Output 'Valid' only if both the chemistry and phases are correct. Output 'Invalid' if the chemistry is wrong or any specified phase is unreasonable.
Print one result per line, preceded by the reaction index.
Do not include any additional text.
1. hydrogen peroxide(aq) + potassium iodide(aq) -> water(l) + iodine(aq) + oxygen(g) [solvent: water]
2. hydrogen peroxide(aq) {acidic solution} + iron(II) sulfate(aq) -> iron(III) sulfate(aq) + water(l) [solvent: water]

--- VALIDATION RESPONSE RECEIVED ---
1. Valid
2. Valid

Validated entries:
[{'cid': 784,
  'confidence': 1.0,
  'positives': 1,
  'reaction': {'catalyst': None,
               'products': [{'name': 'water', 'phase': 'l'},
                            {'name': 'i

## 9. Final Parse Into Graph Reactions

The pipeline parse stage keeps only `valid` reactions, parses names to CIDs, deduplicates by `rid`, and writes parsed LLM reactions when run through the production method. This notebook mirrors that logic in-memory.

In [9]:
parsed_final = []
final_rids = set()
final_skip_stats = {"validator_reject": 0, "invalid_schema": 0, "unmapped": 0, "duplicate_rid": 0}

for entry in validated_entries:
    if not entry.get("valid", False):
        final_skip_stats["validator_reject"] += 1
        continue
    reaction_obj = entry.get("reaction")
    if not _is_valid_reaction_obj(reaction_obj):
        final_skip_stats["invalid_schema"] += 1
        continue
    parsed_reaction, unmapped = chemy.reaction_llm.parse_structured_reaction(reaction_obj)
    if not parsed_reaction:
        final_skip_stats["unmapped"] += 1
        continue
    if parsed_reaction["rid"] in final_rids:
        final_skip_stats["duplicate_rid"] += 1
        continue
    final_rids.add(parsed_reaction["rid"])
    parsed_reaction["confidence"] = entry.get("confidence", 0.0)
    parsed_reaction["source"] = entry.get("source")
    parsed_final.append(parsed_reaction)

print("Final skip stats:")
pprint(final_skip_stats)
print("\nFinal parsed reactions:")
pprint(parsed_final)

Final skip stats:
{'duplicate_rid': 0, 'invalid_schema': 0, 'unmapped': 0, 'validator_reject': 0}

Final parsed reactions:
[{'complexity': 0.4,
  'confidence': 1.0,
  'products': [{'cid': 807,
                'norm_name': 'iodine',
                'original_name': 'iodine',
                'phase': 'aq'},
               {'cid': 962,
                'norm_name': 'water',
                'original_name': 'water',
                'phase': 'l'},
               {'cid': 977,
                'norm_name': 'oxygen',
                'original_name': 'oxygen',
                'phase': 'g'}],
  'reagents': [{'cid': 784,
                'norm_name': 'hydrogenperoxide',
                'original_name': 'hydrogen peroxide',
                'phase': 'aq'},
               {'cid': 4875,
                'norm_name': 'potassiumiodide',
                'original_name': 'potassium iodide',
                'phase': 'aq'}],
  'rid': 'RWk//gUCDWrLfWlrdaTKzA==',
  'solvent': 'water',
  'source': 'demo-validator

## 10. Optional Real Pipeline Execution

This cell is intentionally guarded by `RUN_WRITES`. When enabled, it runs the same production methods used by the CLI for the selected preset. Keep it disabled while inspecting prompts and intermediate behavior.

In [10]:
MAX_WORKERS = 1

if RUN_WRITES:
    if not RUN_REAL_LLM:
        raise RuntimeError("RUN_WRITES requires RUN_REAL_LLM=True; production fetch/validate call the LLM.")
    chemy.raw_reactions.fetch(PRESET_NAME, max_workers=MAX_WORKERS)
    chemy.raw_reactions.validate(PRESET_NAME, max_workers=MAX_WORKERS)
    chemy.raw_reactions.parse([PRESET_NAME])
else:
    print("Dry-run only. To run the real pipeline from the shell:")
    print(f"PYTHONPATH=. python -m scripts.run.fetch_reactions {PRESET_NAME} --workers {MAX_WORKERS}")
    print("\nThen parse accepted verdicts:")
    print("PYTHONPATH=. python - <<'PY'")
    print("from scripts.chemy import Chemy")
    print("chemy = Chemy('data')")
    print(f"chemy.raw_reactions.parse([{PRESET_NAME!r}])")
    print("PY")

Dry-run only. To run the real pipeline from the shell:
PYTHONPATH=. python -m scripts.run.fetch_reactions wiki_crc_rp --workers 1

Then parse accepted verdicts:
PYTHONPATH=. python - <<'PY'
from scripts.chemy import Chemy
chemy = Chemy('data')
chemy.raw_reactions.parse(['wiki_crc_rp'])
PY


## Verification Checklist

Use this notebook to manually inspect these surfaces:

- Fetch prompt asks for clean `name`, required `phase`, and optional omitted `note`.
- Raw LLM response is valid JSONL, one object per line.
- Local parser rejects missing/invalid phases before validation.
- Revalidation prompt asks the model to correct phases and misplaced qualifiers.
- Validation prompt includes `name(phase)` plus `{note}` when note exists.
- CID parsing ignores `phase` and `note` for identity but preserves them in parsed entries.
- Final parsed reactions have stable `rid`, `source`, `confidence`, `reagents`, and `products`.